# Notebook 04 — Enhanced 3D ResUNet Model Architecture

## Purpose

This notebook defines, builds, validates, and saves the **Enhanced 3D ResUNet multi-task architecture** for automatic rib-fracture segmentation and fracture-type classification.

The architecture consumes the patch data produced by Notebook 03:

- CT input: `64 × 64 × 64`
- Input channels: `1`
- Segmentation output: binary fracture mask
- Classification output: five-class softmax

### Classification classes

| Class | Meaning |
|---:|---|
| 0 | Background / negative patch |
| 1 | Displaced fracture |
| 2 | Non-displaced fracture |
| 3 | Buckle fracture |
| 4 | Segmental fracture |

### Important methodological rule

Notebook 04 **does not train the model**. It only defines and validates the architecture. Training, losses, callbacks, augmentation, and optimization belong to Notebook 05.

The model is designed as a **multi-task network** with a shared 3D encoder and two task-specific decoder/heads:

1. **Segmentation branch:** U-Net-style decoder with residual blocks and skip connections.
2. **Classification branch:** global feature aggregation followed by dense layers and a five-class softmax output.


In [ ]:
# Cell 1 — Imports and reproducibility

import os
import json
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass


## 1. Persistent paths

Notebook 03 saves the development patch data under `/kaggle/working/patch_dataset`.

For the full experiment, the extracted patch dataset should be published as a Kaggle Dataset or otherwise mounted persistently. This notebook therefore supports both:

- a persistent mounted dataset path, and
- the local `/kaggle/working/patch_dataset` path used during development.

The model itself is saved under `/kaggle/working/model_artifacts/`.


In [ ]:
# Cell 2 — Locate the patch dataset

from pathlib import Path

CANDIDATE_PATCH_ROOTS = [
    Path("/kaggle/input/datasets/waboke/ribfrac-processed-dev/patch_dataset"),
    Path("/kaggle/input/ribfrac-patch-dataset/patch_dataset"),
    Path("/kaggle/working/patch_dataset"),
]

PATCH_ROOT = next((p for p in CANDIDATE_PATCH_ROOTS if p.exists()), None)

if PATCH_ROOT is None:
    raise FileNotFoundError(
        "Patch dataset not found. Expected one of:\n" +
        "\n".join(str(p) for p in CANDIDATE_PATCH_ROOTS)
    )

MODEL_ROOT = Path("/kaggle/working/model_artifacts")
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

print("PATCH_ROOT:", PATCH_ROOT)
print("MODEL_ROOT:", MODEL_ROOT)


In [ ]:
# Cell 3 — Architecture configuration

PATCH_SIZE = (64, 64, 64)
INPUT_CHANNELS = 1
NUM_CLASSES = 5

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental",
}

BASE_FILTERS = 16
DEPTH = 4
DROPOUT_RATE = 0.20

print("Patch size:", PATCH_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Base filters:", BASE_FILTERS)
print("Encoder depth:", DEPTH)
print("Dropout:", DROPOUT_RATE)


## 2. Load and validate the Notebook 03 development case

Notebook 03 uses `RibFrac128` as an end-to-end pipeline validation case. This case is **not** representative of the full dataset and must not be used to report final model performance.

The validation performed here checks that:

- CT patches have the expected spatial dimensions;
- the segmentation target is binary;
- the classification target contains values in `0–4`;
- all arrays have the same number of patches;
- the CT input has one channel.


In [ ]:
# Cell 4 — Locate a saved patch case

ribfrac128_dir = PATCH_ROOT / "RibFrac128"

if not ribfrac128_dir.exists():
    raise FileNotFoundError(
        f"Expected development case directory not found: {ribfrac128_dir}"
    )

X_PATH = ribfrac128_dir / "X.npy"
Y_SEG_PATH = ribfrac128_dir / "Y_seg.npy"
Y_CLASS_PATH = ribfrac128_dir / "Y_class.npy"

for path in [X_PATH, Y_SEG_PATH, Y_CLASS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

X_dev = np.load(X_PATH, mmap_mode="r")
Y_seg_dev = np.load(Y_SEG_PATH, mmap_mode="r")
Y_class_dev = np.load(Y_CLASS_PATH, mmap_mode="r")

print("X:", X_dev.shape, X_dev.dtype)
print("Y_seg:", Y_seg_dev.shape, Y_seg_dev.dtype)
print("Y_class:", Y_class_dev.shape, Y_class_dev.dtype)


In [ ]:
# Cell 5 — Validate development-case arrays

assert X_dev.ndim in (4, 5), f"Unexpected X dimensions: {X_dev.shape}"
assert Y_seg_dev.ndim in (4, 5), f"Unexpected Y_seg dimensions: {Y_seg_dev.shape}"
assert Y_class_dev.ndim in (1, 2), f"Unexpected Y_class dimensions: {Y_class_dev.shape}"

assert X_dev.shape[0] == Y_seg_dev.shape[0] == Y_class_dev.shape[0]

# Convert channel-last/channel-free development arrays conceptually to patch geometry.
if X_dev.ndim == 4:
    assert tuple(X_dev.shape[1:]) == PATCH_SIZE
else:
    assert tuple(X_dev.shape[1:4]) == PATCH_SIZE
    assert X_dev.shape[-1] == INPUT_CHANNELS

if Y_seg_dev.ndim == 4:
    assert tuple(Y_seg_dev.shape[1:]) == PATCH_SIZE
else:
    assert tuple(Y_seg_dev.shape[1:4]) == PATCH_SIZE
    assert Y_seg_dev.shape[-1] == 1

class_values = np.unique(Y_class_dev)
seg_values = np.unique(Y_seg_dev)

assert set(seg_values.tolist()).issubset({0, 1}), seg_values
assert set(class_values.tolist()).issubset(set(range(NUM_CLASSES))), class_values

print("Array validation: PASS")
print("Number of development patches:", X_dev.shape[0])
print("Segmentation values:", seg_values)
print("Classification values:", class_values)


## 3. Input-shape normalization

Keras will use channel-last tensors:

```text
(batch, 64, 64, 64, 1)
```

Notebook 03 may save arrays without the final channel dimension. The helper below adds the channel dimension when necessary. It does not alter the voxel values.


In [ ]:
# Cell 6 — Standardize input tensor shapes

def ensure_channel_last_x(x):
    x = np.asarray(x)
    if x.ndim == 4:
        return x[..., np.newaxis]
    if x.ndim == 5 and x.shape[-1] == 1:
        return x
    raise ValueError(f"Unsupported X shape: {x.shape}")

def ensure_channel_last_seg(y):
    y = np.asarray(y)
    if y.ndim == 4:
        return y[..., np.newaxis]
    if y.ndim == 5 and y.shape[-1] == 1:
        return y
    raise ValueError(f"Unsupported segmentation shape: {y.shape}")

X_dev_ch = ensure_channel_last_x(X_dev)
Y_seg_dev_ch = ensure_channel_last_seg(Y_seg_dev)

print("Standardized X:", X_dev_ch.shape)
print("Standardized Y_seg:", Y_seg_dev_ch.shape)


# 4. Enhanced 3D ResUNet architecture

The model follows a residual U-Net design:

```text
                 ┌──────────── Encoder ────────────┐
Input 3D CT ───► ResBlock ─► Down ─► ResBlock ─► Down ─►
                                                     │
                                                   Bottleneck
                                                     │
                           ┌──────── Classification head
                           │
                           └──────── Decoder ◄─────────────
                                      │
                                  Skip connections
                                      │
                               Segmentation output
```

### Architectural principles

**Residual learning:** each residual block learns a transformation added to an identity/projection shortcut. This improves optimization of the deeper 3D network.

**U-Net skip connections:** encoder feature maps are concatenated with decoder features at corresponding resolutions to recover spatial information.

**3D convolution:** all convolutional operations operate directly on the volumetric CT patches.

**Multi-task learning:** segmentation and classification are learned from a shared representation but have separate output heads.

### Classification branch

The classification branch uses the deepest shared representation and applies:

1. global average pooling;
2. dense feature transformation;
3. dropout;
4. five-class softmax.

This produces a probability distribution over the five patch classes.


In [ ]:
# Cell 7 — Residual 3D convolutional block

def residual_block_3d(
    x,
    filters,
    name,
    dropout_rate=0.0,
):
    shortcut = x

    x = layers.Conv3D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_conv1",
    )(x)
    x = layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = layers.Activation("relu", name=f"{name}_relu1")(x)

    x = layers.Conv3D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_conv2",
    )(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x)

    # Projection shortcut when the channel dimension changes.
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(
            filters,
            kernel_size=1,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"{name}_proj",
        )(shortcut)
        shortcut = layers.BatchNormalization(
            name=f"{name}_proj_bn"
        )(shortcut)

    x = layers.Add(name=f"{name}_add")([x, shortcut])
    x = layers.Activation("relu", name=f"{name}_out")(x)

    if dropout_rate > 0:
        x = layers.Dropout(
            dropout_rate,
            name=f"{name}_dropout"
        )(x)

    return x


In [ ]:
# Cell 8 — Encoder downsampling block

def downsample_block_3d(x, filters, name):
    return layers.Conv3D(
        filters,
        kernel_size=2,
        strides=2,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=name,
    )(x)


In [ ]:
# Cell 9 — Decoder block

def decoder_block_3d(x, skip, filters, name, dropout_rate=0.0):
    x = layers.Conv3DTranspose(
        filters,
        kernel_size=2,
        strides=2,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_up",
    )(x)

    x = layers.Concatenate(name=f"{name}_concat")([x, skip])

    x = residual_block_3d(
        x,
        filters,
        name=f"{name}_res1",
        dropout_rate=dropout_rate,
    )

    return x


## 5. Build the complete multi-task model

The default configuration uses four encoder resolutions:

- Level 1: 16 filters
- Level 2: 32 filters
- Level 3: 64 filters
- Level 4: 128 filters
- Bottleneck: 256 filters

The decoder returns to the original `64³` spatial resolution.

The exact filter width can be reduced later if GPU memory becomes a limiting factor. The architectural definition is kept in one function so the configuration can be changed reproducibly.


In [ ]:
# Cell 10 — Build Enhanced 3D ResUNet

def build_enhanced_3d_resunet(
    input_shape=(64, 64, 64, 1),
    num_classes=5,
    base_filters=16,
    dropout_rate=0.20,
):
    inputs = keras.Input(
        shape=input_shape,
        name="ct_input",
    )

    # -------------------------
    # Encoder
    # -------------------------
    e1 = residual_block_3d(
        inputs,
        base_filters,
        name="enc1",
        dropout_rate=dropout_rate,
    )

    d1 = downsample_block_3d(
        e1,
        base_filters * 2,
        name="down1",
    )

    e2 = residual_block_3d(
        d1,
        base_filters * 2,
        name="enc2",
        dropout_rate=dropout_rate,
    )

    d2 = downsample_block_3d(
        e2,
        base_filters * 4,
        name="down2",
    )

    e3 = residual_block_3d(
        d2,
        base_filters * 4,
        name="enc3",
        dropout_rate=dropout_rate,
    )

    d3 = downsample_block_3d(
        e3,
        base_filters * 8,
        name="down3",
    )

    e4 = residual_block_3d(
        d3,
        base_filters * 8,
        name="enc4",
        dropout_rate=dropout_rate,
    )

    d4 = downsample_block_3d(
        e4,
        base_filters * 16,
        name="down4",
    )

    bottleneck = residual_block_3d(
        d4,
        base_filters * 16,
        name="bottleneck",
        dropout_rate=dropout_rate,
    )

    # -------------------------
    # Classification branch
    # -------------------------
    cls = layers.GlobalAveragePooling3D(
        name="classification_gap"
    )(bottleneck)

    cls = layers.Dense(
        base_filters * 8,
        activation="relu",
        kernel_initializer="he_normal",
        name="classification_dense",
    )(cls)

    cls = layers.Dropout(
        dropout_rate,
        name="classification_dropout",
    )(cls)

    classification_output = layers.Dense(
        num_classes,
        activation="softmax",
        name="classification_output",
    )(cls)

    # -------------------------
    # Segmentation decoder
    # -------------------------
    x = decoder_block_3d(
        bottleneck, e4,
        base_filters * 8,
        name="dec4",
        dropout_rate=dropout_rate,
    )

    x = decoder_block_3d(
        x, e3,
        base_filters * 4,
        name="dec3",
        dropout_rate=dropout_rate,
    )

    x = decoder_block_3d(
        x, e2,
        base_filters * 2,
        name="dec2",
        dropout_rate=dropout_rate,
    )

    x = decoder_block_3d(
        x, e1,
        base_filters,
        name="dec1",
        dropout_rate=dropout_rate,
    )

    segmentation_output = layers.Conv3D(
        1,
        kernel_size=1,
        activation="sigmoid",
        padding="same",
        name="segmentation_output",
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs={
            "segmentation_output": segmentation_output,
            "classification_output": classification_output,
        },
        name="Enhanced_3D_ResUNet_MultiTask",
    )

    return model


In [ ]:
# Cell 11 — Instantiate the model

model = build_enhanced_3d_resunet(
    input_shape=(*PATCH_SIZE, INPUT_CHANNELS),
    num_classes=NUM_CLASSES,
    base_filters=BASE_FILTERS,
    dropout_rate=DROPOUT_RATE,
)

model.summary()


## 6. Architecture validation

Before any training is attempted, the model must satisfy four structural conditions:

1. Input is `64 × 64 × 64 × 1`.
2. Segmentation output is `64 × 64 × 64 × 1`.
3. Classification output has five probabilities.
4. The classification probabilities sum to one for each sample.



In [ ]:
# Cell 12 — Verify model output shapes

input_shape_expected = (None, 64, 64, 64, 1)
seg_shape_expected = (None, 64, 64, 64, 1)
cls_shape_expected = (None, 5)

assert model.input_shape == input_shape_expected, model.input_shape

assert model.output_shape["segmentation_output"] == seg_shape_expected
assert model.output_shape["classification_output"] == cls_shape_expected

print("Input shape:", model.input_shape)
print("Segmentation output:", model.output_shape["segmentation_output"])
print("Classification output:", model.output_shape["classification_output"])
print("Shape validation: PASS")


In [ ]:
# Cell 13 — Forward-pass test

# Use a very small batch to verify that the complete graph executes.
test_batch_size = min(2, X_dev_ch.shape[0])

x_test = np.asarray(X_dev_ch[:test_batch_size], dtype=np.float32)

outputs = model(x_test, training=False)

seg_pred = outputs["segmentation_output"].numpy()
cls_pred = outputs["classification_output"].numpy()

assert seg_pred.shape == (test_batch_size, 64, 64, 64, 1)
assert cls_pred.shape == (test_batch_size, NUM_CLASSES)

assert np.all(seg_pred >= 0.0)
assert np.all(seg_pred <= 1.0)

cls_sums = cls_pred.sum(axis=1)
assert np.allclose(cls_sums, 1.0, atol=1e-5)

print("Forward pass: PASS")
print("Segmentation prediction shape:", seg_pred.shape)
print("Classification prediction shape:", cls_pred.shape)
print("Classification probability sums:", cls_sums)


## 7. Compile configuration for Notebook 05

The model has two outputs and therefore requires a loss for each task.

### Segmentation

Binary cross-entropy is used as the baseline segmentation loss. Notebook 05 can introduce a combined loss such as BCE + Dice loss if the class imbalance requires it.

### Classification

Sparse categorical cross-entropy is appropriate because Notebook 03 stores the class target as an integer class index (`0–4`), rather than a one-hot vector.

### Important

Compilation here does **not** constitute training. The final loss weights and optimization strategy should be experimentally justified in Notebook 05.


In [ ]:
# Cell 14 — Baseline multi-task compilation

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss={
        "segmentation_output": keras.losses.BinaryCrossentropy(),
        "classification_output": keras.losses.SparseCategoricalCrossentropy(),
    },
    loss_weights={
        "segmentation_output": 1.0,
        "classification_output": 1.0,
    },
    metrics={
        "segmentation_output": [
            keras.metrics.BinaryAccuracy(name="seg_accuracy"),
        ],
        "classification_output": [
            keras.metrics.SparseCategoricalAccuracy(name="cls_accuracy"),
        ],
    },
)

print("Model compilation: PASS")


In [ ]:
# Cell 15 — One-batch loss/gradient smoke test

# This is NOT model training. It only checks that the compiled multi-task
# model can calculate its losses and gradients successfully.

y_seg_test = np.asarray(
    Y_seg_dev_ch[:test_batch_size],
    dtype=np.float32,
)

y_cls_test = np.asarray(
    Y_class_dev[:test_batch_size],
    dtype=np.int32,
).reshape(-1)

with tf.GradientTape() as tape:
    pred = model(x_test, training=True)

    total_loss = model.compiled_loss(
        {
            "segmentation_output": y_seg_test,
            "classification_output": y_cls_test,
        },
        pred,
        regularization_losses=model.losses,
    )

gradients = tape.gradient(total_loss, model.trainable_variables)

valid_gradients = [
    g for g in gradients
    if g is not None
]

assert len(valid_gradients) == len(model.trainable_variables)
assert all(
    bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
    for g in valid_gradients
)

print("Loss/gradient smoke test: PASS")
print("Total loss:", float(total_loss.numpy()))
print("Trainable variables:", len(model.trainable_variables))


## 8. Save the architecture

The model is saved in Keras format together with a JSON configuration. This makes the architecture reproducible and allows Notebook 05 to load the exact model definition rather than rebuilding it manually.



In [ ]:
# Cell 16 — Save model and architecture configuration

MODEL_PATH = MODEL_ROOT / "enhanced_3d_resunet_multitask.keras"
CONFIG_PATH = MODEL_ROOT / "model_config.json"

model.save(MODEL_PATH)

model_config = {
    "model_name": "Enhanced_3D_ResUNet_MultiTask",
    "input_shape": [64, 64, 64, 1],
    "patch_size": [64, 64, 64],
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "base_filters": BASE_FILTERS,
    "depth": DEPTH,
    "dropout_rate": DROPOUT_RATE,
    "segmentation_output": {
        "channels": 1,
        "activation": "sigmoid",
        "task": "binary fracture segmentation",
    },
    "classification_output": {
        "classes": NUM_CLASSES,
        "activation": "softmax",
        "task": "five-class patch classification",
    },
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "losses": {
        "segmentation_output": "BinaryCrossentropy",
        "classification_output": "SparseCategoricalCrossentropy",
    },
    "loss_weights": {
        "segmentation_output": 1.0,
        "classification_output": 1.0,
    },
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=2)

print("Model saved:", MODEL_PATH)
print("Configuration saved:", CONFIG_PATH)


In [ ]:
# Cell 17 — Reload saved model

reloaded_model = keras.models.load_model(MODEL_PATH)

assert reloaded_model.input_shape == model.input_shape
assert reloaded_model.output_shape == model.output_shape

reloaded_outputs = reloaded_model(
    x_test,
    training=False,
)

assert reloaded_outputs["segmentation_output"].shape == seg_pred.shape
assert reloaded_outputs["classification_output"].shape == cls_pred.shape

print("Reload validation: PASS")
print("Reloaded model:", reloaded_model.name)


# Notebook 04 completion criteria

Notebook 04 is complete when all of the following are satisfied:

- [ ] Notebook 03 patch data can be located.
- [ ] Development-case arrays pass shape/value validation.
- [ ] Input tensor is `64×64×64×1`.
- [ ] Residual 3D encoder is constructed.
- [ ] U-Net skip connections are constructed.
- [ ] Binary segmentation head is constructed.
- [ ] Five-class softmax classification head is constructed.
- [ ] Forward pass succeeds.
- [ ] Classification probabilities sum to one.
- [ ] Multi-task losses can be calculated.
- [ ] Gradients are finite.
- [ ] Model saves successfully.
- [ ] Saved model reloads successfully.

## Important interpretation

Successful execution of this notebook demonstrates **architectural and computational validity**. It does **not** demonstrate clinical or predictive performance.

Notebook 05 will address:

1. full train/validation patch loading;
2. class imbalance;
3. segmentation loss design;
4. classification loss;
5. multi-task loss weighting;
6. data augmentation;
7. optimizer and learning-rate scheduling;
8. checkpointing and early stopping;
9. training history;
10. reproducible experiment logging.
